# Notebook #11: Single Cell Cell-Cell Communication (sc-CCC)

This notebook LIANA/CellPhoneDB to characterize immune cell-cell communication across endometriosis tissue types and to evaluate communication involving senescent/dysfunctional immune states.

###Main Questions:
1. What does CCC look like in lesions vs control tissues?
2. Are there specific cells that are communication hubs?
3. Do senescent, dysfunctional cells communicate differently varying on tissue types?
4. Which pathways are enriched in each analysis?


In [1]:
!pip install -q \
    numpy==2.0.2 \
    pandas==2.3.2 \
    pyarrow==18.1.0 \
    anndata==0.12.6 \
    scanpy==1.11.5 \
    liana==1.7.3 \
    matplotlib \
    seaborn \
    plotnine \
    gprofiler-official \
    igraph

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 6.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 91.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.3/172.3 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 78.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 563.9/563.9 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 87.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.7/363.7 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 122.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency confl

In [2]:
# -- Imports
from pathlib import Path

import pandas as pd
import numpy as np
import seaborn as sns

import matplotlib.pyplot as plt
import liana as li
import scanpy as sc
from liana.method import cellphonedb
from IPython.display import display
from plotnine import ggtitle, labs
from pathlib import Path
from gprofiler import GProfiler

import anndata as ad
ad.settings.allow_write_nullable_strings = True

In [3]:
from google.colab import drive
drive.mount('/content/drive', force_remount = True)

Mounted at /content/drive


In [4]:
# -- Paths
dataset = "GSE179640"

project_dir = Path(
    "/content/drive/MyDrive/endo-immune-atlas"
)

input_file = (
    project_dir
    / "data"
    / "interim"
    / "GSE179640"
    / "immunosenescence.h5ad"
)

results_path = (
    project_dir
    / "results"
    / dataset
    / "cell_communication"
)

figures_path = (
    project_dir
    / "figures"
    / dataset
    / "cell_communication"
)

all_immune_results_path = results_path / "all_immune"
all_immune_figures_path = figures_path / "all_immune"
all_immune_pathway_path = all_immune_results_path / "pathway_enrichment"

sen_dys_results_path = results_path / "sen_dys"
sen_dys_figures_path = figures_path / "sen_dys"
sen_dys_pathway_path = sen_dys_results_path / "pathway_enrichment"

for path in [
    all_immune_results_path,
    all_immune_figures_path,
    all_immune_pathway_path,
    sen_dys_results_path,
    sen_dys_figures_path,
    sen_dys_pathway_path,
]:
    path.mkdir(parents=True, exist_ok=True)

In [5]:
# -- Parameters
P_VALUE_THRESHOLD = 0.05
LR_MEANS_THRESHOLD = 0.5
MIN_CELLS = 20
TOP_N = 20

TISSUE_ORDER = [
    "Ctrl",
    "EuE",
    "EcO",
    "EcP"
]

In [6]:
# -- Import objects
immune_subset = sc.read_h5ad(
    input_file
)
required_obs_columns = {
    "tissue_type",
    "immune_cell_type",
    "cell_type_short",
    "sen_dysfunction_label"
}

missing_columns = (
    required_obs_columns
    - set(immune_subset.obs.columns)
)

if missing_columns:
    raise KeyError(
        "Missing required obs columns: "
        f"{sorted(missing_columns)}"
    )

if "Proliferating" in (
    immune_subset.obs["immune_cell_type"]
    .astype(str)
    .unique()
):
    immune_subset = immune_subset[
        immune_subset.obs["immune_cell_type"]
        .astype(str)
        != "Proliferating"
    ].copy()

In [7]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# FUNCTIONS
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

def make_tissue_subsets(
    adata,
    tissue_col="tissue_type",
    tissue_order=None
):
    observed_tissues = (
        adata.obs[tissue_col]
        .astype(str)
        .unique()
        .tolist()
    )

    tissues = (
        observed_tissues
        if tissue_order is None
        else [
            tissue
            for tissue in tissue_order
            if tissue in observed_tissues
        ]
    )

    return {
        tissue: adata[
            adata.obs[tissue_col].astype(str) == tissue
        ].copy()
        for tissue in tissues
    }


def filter_minimum_groups(
    adata,
    groupby,
    min_cells=MIN_CELLS
):
    counts = adata.obs[groupby].value_counts()

    valid_groups = counts[
        counts >= min_cells
    ].index

    return adata[
        adata.obs[groupby].isin(valid_groups)
    ].copy()


def run_cellphonedb_by_tissue(
    subsets,
    groupby,
    use_raw=False,
    layer=None,
    min_cells=MIN_CELLS,
    n_perms=1000,
    seed=1337
):
    results = {}

    for tissue, subset in subsets.items():

        subset = filter_minimum_groups(
            subset,
            groupby=groupby,
            min_cells=min_cells
        )

        n_groups = subset.obs[groupby].nunique()

        if n_groups < 2:
            print(
                f"Skipping {tissue}: "
                f"only {n_groups} valid group(s)."
            )
            continue

        print(
            f"Running CellPhoneDB for {tissue}: "
            f"{subset.n_obs:,} cells, "
            f"{n_groups} groups"
        )

        li.mt.cellphonedb(
            subset,
            groupby=groupby,
            resource_name="consensus",
            expr_prop=0.1,
            min_cells=min_cells,
            use_raw=use_raw,
            layer=layer,
            n_perms=n_perms,
            seed=seed,
            verbose=False
        )

        results[tissue] = subset

        print(
            f"Completed {tissue}: "
            f"{len(subset.uns['liana_res']):,} tested interactions"
        )

    return results


def get_significant_lr_interactions(
    adata,
    p_val_threshold=P_VALUE_THRESHOLD,
    lr_means_threshold=LR_MEANS_THRESHOLD,
    columns=None
):
    if "liana_res" not in adata.uns:
        raise KeyError(
            "LIANA results are missing from adata.uns['liana_res']."
        )

    significant = (
        adata.uns["liana_res"]
        .copy()
        .loc[
            lambda frame: (
                frame["cellphone_pvals"] < p_val_threshold
            )
            & (
                frame["lr_means"] >= lr_means_threshold
            )
        ]
        .sort_values(
            "lr_means",
            ascending=False
        )
        .reset_index(drop=True)
    )

    if columns is not None:
        significant = significant[columns]

    return significant


def save_significant_results(
    subsets,
    output_path,
    file_suffix
):
    significant_results = {}

    for tissue, subset in subsets.items():

        significant = get_significant_lr_interactions(
            subset
        )

        significant_results[tissue] = significant

        significant.to_csv(
            output_path
            / f"{tissue}_{file_suffix}.csv",
            index=False
        )

        print(
            f"{tissue}: "
            f"{len(significant):,} significant interactions"
        )

    return significant_results


def extract_lr_genes(
    significant_results
):
    lr_genes = {}

    for tissue, results in significant_results.items():

        genes = sorted(
            set(results["ligand"].dropna())
            | set(results["receptor"].dropna())
        )

        lr_genes[tissue] = genes

        print(
            f"{tissue}: "
            f"{len(genes)} unique LR genes"
        )

    return lr_genes


def run_pathway_enrichment(
    lr_genes,
    output_path,
    file_suffix
):
    gprofiler = GProfiler(
        return_dataframe=True
    )

    pathway_results = {}

    for tissue, genes in lr_genes.items():

        if not genes:
            print(
                f"Skipping {tissue}: no LR genes."
            )
            pathway_results[tissue] = pd.DataFrame()
            continue

        print(
            f"Running pathway enrichment: {tissue}"
        )

        results = gprofiler.profile(
            organism="hsapiens",
            query=genes
        )

        pathway_results[tissue] = results

        results.to_csv(
            output_path
            / f"{tissue}_{file_suffix}.csv",
            index=False
        )

    return pathway_results


def plot_cross_tissue_enrichment(
    pathway_results,
    output_file,
    title,
    top_n=8,
    tissue_order=TISSUE_ORDER
):
    combined = []

    for tissue, results in pathway_results.items():

        if results is None or results.empty:
            continue

        tissue_results = results.copy()
        tissue_results["tissue"] = tissue
        combined.append(tissue_results)

    if not combined:
        print(
            "No enrichment results available to plot."
        )
        return

    combined_df = pd.concat(
        combined,
        ignore_index=True
    )

    combined_df["neg_log_p"] = -np.log10(
        combined_df["p_value"].clip(
            lower=np.finfo(float).tiny
        )
    )

    top_terms = (
        combined_df
        .sort_values("p_value")
        .groupby(
            "tissue",
            observed=True
        )
        .head(top_n)["name"]
        .unique()
    )

    plot_df = combined_df[
        combined_df["name"].isin(top_terms)
    ].copy()

    term_order = (
        plot_df
        .groupby("name")["neg_log_p"]
        .max()
        .sort_values(ascending=False)
        .index
        .tolist()
    )

    observed_tissues = [
        tissue
        for tissue in tissue_order
        if tissue in plot_df["tissue"].unique()
    ]

    plot_df["name"] = pd.Categorical(
        plot_df["name"],
        categories=term_order,
        ordered=True
    )

    plot_df["tissue"] = pd.Categorical(
        plot_df["tissue"],
        categories=observed_tissues,
        ordered=True
    )

    fig, ax = plt.subplots(
        figsize=(
            1.5 * len(observed_tissues) + 3,
            max(6, 0.45 * len(term_order) + 2)
        )
    )

    scatter = ax.scatter(
        x=plot_df["tissue"].cat.codes,
        y=plot_df["name"].cat.codes,
        s=plot_df["intersection_size"],
        c=plot_df["neg_log_p"],
        cmap="viridis",
        edgecolors="black",
        linewidths=0.3
    )

    ax.set_xticks(
        range(len(observed_tissues))
    )

    ax.set_xticklabels(
        observed_tissues,
        rotation=45,
        ha="right"
    )

    ax.set_yticks(
        range(len(term_order))
    )

    ax.set_yticklabels(
        term_order,
        fontsize=8
    )

    ax.set_xlabel("Tissue")
    ax.set_ylabel("Enriched pathway")
    ax.set_title(title)

    colorbar = plt.colorbar(
        scatter,
        ax=ax
    )

    colorbar.set_label(
        "-log10(p-value)"
    )

    overlap_sizes = [25, 50, 100]

    handles = [
        plt.scatter(
            [],
            [],
            s=size,
            color="gray",
            edgecolors="black"
        )
        for size in overlap_sizes
    ]

    ax.legend(
        handles,
        [str(size) for size in overlap_sizes],
        title="Gene overlap",
        loc="upper left",
        bbox_to_anchor=(1.25, 1)
    )

    plt.tight_layout()

    plt.savefig(
        output_file,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()


def save_liana_plot(
    plot,
    output_path,
    filename,
    width,
    height
):
    plot.save(
        output_path / filename,
        width=width,
        height=height,
        dpi=300,
        verbose=False
    )

    plt.close("all")

### Analysis 1: Communication among all immune cell types

CellPhoneDB is run independently in each tissue so that inferred interactions are tissue-specific.


In [8]:
all_cell_subsets = make_tissue_subsets(
    immune_subset,
    tissue_order=TISSUE_ORDER
)

all_immune_subsets = run_cellphonedb_by_tissue(
    all_cell_subsets,
    groupby="cell_type_short",
    use_raw=False,
    layer=None,
    min_cells=MIN_CELLS
)

Running CellPhoneDB for Ctrl: 2,943 cells, 11 groups


/usr/local/lib/python3.12/dist-packages/legacy_api_wrap/__init__.py:88: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
/usr/local/lib/python3.12/dist-packages/liana/method/_pipe_utils/_pre.py:165: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
/usr/local/lib/python3.12/dist-packages/liana/method/_pipe_utils/_pre.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


Completed Ctrl: 20,612 tested interactions
Running CellPhoneDB for EuE: 9,313 cells, 13 groups


/usr/local/lib/python3.12/dist-packages/legacy_api_wrap/__init__.py:88: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
/usr/local/lib/python3.12/dist-packages/liana/method/_pipe_utils/_pre.py:165: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
/usr/local/lib/python3.12/dist-packages/liana/method/_pipe_utils/_pre.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


Completed EuE: 37,887 tested interactions
Running CellPhoneDB for EcO: 4,063 cells, 13 groups


/usr/local/lib/python3.12/dist-packages/legacy_api_wrap/__init__.py:88: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
/usr/local/lib/python3.12/dist-packages/liana/method/_pipe_utils/_pre.py:165: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
/usr/local/lib/python3.12/dist-packages/liana/method/_pipe_utils/_pre.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


Completed EcO: 32,092 tested interactions
Running CellPhoneDB for EcP: 11,532 cells, 13 groups


/usr/local/lib/python3.12/dist-packages/legacy_api_wrap/__init__.py:88: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
/usr/local/lib/python3.12/dist-packages/liana/method/_pipe_utils/_pre.py:165: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
/usr/local/lib/python3.12/dist-packages/liana/method/_pipe_utils/_pre.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


Completed EcP: 33,691 tested interactions


In [9]:
# -- All-immune LIANA plots
senders = [
    "Mono-C",
    "Mono-NC",
    "TRM",
    "cDC1",
    "cDC2",
    "pDC",
    "NK-CD16+",
    "NK-CD16-"
]

receivers = [
    "Mono-C",
    "Mono-NC",
    "TRM",
    "cDC1",
    "cDC2",
    "pDC",
    "NK-CD16+",
    "NK-CD16-",
    "CD4 T",
    "CD8 T",
    "Treg",
    "γδ T",
    "B"
]

available_cell_types = set(
    immune_subset.obs["cell_type_short"]
    .astype(str)
    .unique()
)

senders = [
    label
    for label in senders
    if label in available_cell_types
]

receivers = [
    label
    for label in receivers
    if label in available_cell_types
]

print("Senders:", senders)
print("Receivers:", receivers)

Senders: ['Mono-C', 'Mono-NC', 'TRM', 'cDC1', 'cDC2', 'pDC', 'NK-CD16+', 'NK-CD16-']
Receivers: ['Mono-C', 'Mono-NC', 'TRM', 'cDC1', 'cDC2', 'pDC', 'NK-CD16+', 'NK-CD16-', 'CD4 T', 'CD8 T', 'Treg', 'γδ T', 'B']


In [10]:
# -- Tile plots
for tissue, subset in all_immune_subsets.items():

    liana_results = subset.uns["liana_res"]

    available_sources = set(
        liana_results["source"].astype(str).unique()
    )

    available_targets = set(
        liana_results["target"].astype(str).unique()
    )

    tissue_senders = [
        label
        for label in senders
        if label in available_sources
    ]

    tissue_receivers = [
        label
        for label in receivers
        if label in available_targets
    ]

    print(
        f"{tissue} senders:",
        tissue_senders
    )

    print(
        f"{tissue} receivers:",
        tissue_receivers
    )

    if not tissue_senders or not tissue_receivers:
        print(
            f"Skipping {tissue}: no requested "
            "sender-receiver groups are available."
        )
        continue

    tile_plot = li.pl.tileplot(
        adata=subset,
        fill="means",
        label="props",
        label_fun=lambda value: f"{value:.2f}",
        top_n=TOP_N,
        orderby="lr_means",
        orderby_ascending=False,
        source_labels=tissue_senders,
        source_title="Sender",
        target_labels=tissue_receivers,
        target_title="Receiver",
        filter_fun=lambda frame: (
            frame["cellphone_pvals"] < P_VALUE_THRESHOLD
        )
        & (
            frame["lr_means"] >= LR_MEANS_THRESHOLD
        ),
        figure_size=(20, 20)
    )

    tile_plot = (
        tile_plot
        + ggtitle(
            f"All immune CCC — {tissue}"
        )
    )

    save_liana_plot(
        tile_plot,
        output_path=all_immune_figures_path,
        filename=f"{tissue}_all_immune_tileplot.png",
        width=20,
        height=20
    )

Ctrl senders: ['Mono-C', 'Mono-NC', 'TRM', 'cDC2', 'NK-CD16+', 'NK-CD16-']
Ctrl receivers: ['Mono-C', 'Mono-NC', 'TRM', 'cDC2', 'NK-CD16+', 'NK-CD16-', 'CD4 T', 'CD8 T', 'Treg', 'γδ T', 'B']


/usr/local/lib/python3.12/dist-packages/liana/plotting/_common.py:108: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


EuE senders: ['Mono-C', 'Mono-NC', 'TRM', 'cDC1', 'cDC2', 'pDC', 'NK-CD16+', 'NK-CD16-']
EuE receivers: ['Mono-C', 'Mono-NC', 'TRM', 'cDC1', 'cDC2', 'pDC', 'NK-CD16+', 'NK-CD16-', 'CD4 T', 'CD8 T', 'Treg', 'γδ T', 'B']


/usr/local/lib/python3.12/dist-packages/liana/plotting/_common.py:108: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


EcO senders: ['Mono-C', 'Mono-NC', 'TRM', 'cDC1', 'cDC2', 'pDC', 'NK-CD16+', 'NK-CD16-']
EcO receivers: ['Mono-C', 'Mono-NC', 'TRM', 'cDC1', 'cDC2', 'pDC', 'NK-CD16+', 'NK-CD16-', 'CD4 T', 'CD8 T', 'Treg', 'γδ T', 'B']


/usr/local/lib/python3.12/dist-packages/liana/plotting/_common.py:108: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


EcP senders: ['Mono-C', 'Mono-NC', 'TRM', 'cDC1', 'cDC2', 'pDC', 'NK-CD16+', 'NK-CD16-']
EcP receivers: ['Mono-C', 'Mono-NC', 'TRM', 'cDC1', 'cDC2', 'pDC', 'NK-CD16+', 'NK-CD16-', 'CD4 T', 'CD8 T', 'Treg', 'γδ T', 'B']


/usr/local/lib/python3.12/dist-packages/liana/plotting/_common.py:108: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


In [11]:
# -- Dot plots
for tissue, subset in all_immune_subsets.items():

    liana_results = subset.uns["liana_res"]

    available_sources = set(
        liana_results["source"]
        .astype(str)
        .unique()
    )

    available_targets = set(
        liana_results["target"]
        .astype(str)
        .unique()
    )

    tissue_senders = [
        label
        for label in senders
        if label in available_sources
    ]

    tissue_receivers = [
        label
        for label in receivers
        if label in available_targets
    ]

    print(
        f"{tissue} senders:",
        tissue_senders
    )

    print(
        f"{tissue} receivers:",
        tissue_receivers
    )

    if not tissue_senders or not tissue_receivers:
        print(
            f"Skipping {tissue}: no requested "
            "sender-receiver groups are available."
        )
        continue

    dot_plot = li.pl.dotplot(
        adata=subset,
        colour="lr_means",
        size="cellphone_pvals",
        inverse_size=True,
        top_n=TOP_N,
        orderby="cellphone_pvals",
        orderby_ascending=True,
        source_labels=tissue_senders,
        target_labels=tissue_receivers,
        filter_fun=lambda frame: (
            frame["cellphone_pvals"] < P_VALUE_THRESHOLD
        )
        & (
            frame["lr_means"] >= LR_MEANS_THRESHOLD
        ),
        figure_size=(22, 22)
    )

    dot_plot = (
        dot_plot
        + ggtitle(
            f"All immune CCC — {tissue}"
        )
        + labs(
            colour="Mean expression",
            size="p-value"
        )
    )

    save_liana_plot(
        dot_plot,
        output_path=all_immune_figures_path,
        filename=f"{tissue}_all_immune_dotplot.png",
        width=22,
        height=22
    )

Ctrl senders: ['Mono-C', 'Mono-NC', 'TRM', 'cDC2', 'NK-CD16+', 'NK-CD16-']
Ctrl receivers: ['Mono-C', 'Mono-NC', 'TRM', 'cDC2', 'NK-CD16+', 'NK-CD16-', 'CD4 T', 'CD8 T', 'Treg', 'γδ T', 'B']


/usr/local/lib/python3.12/dist-packages/liana/plotting/_common.py:108: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


EuE senders: ['Mono-C', 'Mono-NC', 'TRM', 'cDC1', 'cDC2', 'pDC', 'NK-CD16+', 'NK-CD16-']
EuE receivers: ['Mono-C', 'Mono-NC', 'TRM', 'cDC1', 'cDC2', 'pDC', 'NK-CD16+', 'NK-CD16-', 'CD4 T', 'CD8 T', 'Treg', 'γδ T', 'B']


/usr/local/lib/python3.12/dist-packages/liana/plotting/_common.py:108: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


EcO senders: ['Mono-C', 'Mono-NC', 'TRM', 'cDC1', 'cDC2', 'pDC', 'NK-CD16+', 'NK-CD16-']
EcO receivers: ['Mono-C', 'Mono-NC', 'TRM', 'cDC1', 'cDC2', 'pDC', 'NK-CD16+', 'NK-CD16-', 'CD4 T', 'CD8 T', 'Treg', 'γδ T', 'B']


/usr/local/lib/python3.12/dist-packages/liana/plotting/_common.py:108: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


EcP senders: ['Mono-C', 'Mono-NC', 'TRM', 'cDC1', 'cDC2', 'pDC', 'NK-CD16+', 'NK-CD16-']
EcP receivers: ['Mono-C', 'Mono-NC', 'TRM', 'cDC1', 'cDC2', 'pDC', 'NK-CD16+', 'NK-CD16-', 'CD4 T', 'CD8 T', 'Treg', 'γδ T', 'B']


/usr/local/lib/python3.12/dist-packages/liana/plotting/_common.py:108: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


In [12]:
# -- Significant all-immune interactions

significant_all_immune = save_significant_results(
    all_immune_subsets,
    output_path=all_immune_results_path,
    file_suffix="all_immune_significant_lr_pairs"
)

all_immune_lr_genes = extract_lr_genes(
    significant_all_immune
)

Ctrl: 7,229 significant interactions
EuE: 12,637 significant interactions
EcO: 10,302 significant interactions
EcP: 10,731 significant interactions
Ctrl: 400 unique LR genes
EuE: 506 unique LR genes
EcO: 468 unique LR genes
EcP: 489 unique LR genes


In [13]:
# -- Pathway enrichment
all_immune_pathways = run_pathway_enrichment(
    all_immune_lr_genes,
    output_path=all_immune_pathway_path,
    file_suffix="all_immune_pathway_enrichment"
)

plot_cross_tissue_enrichment(
    all_immune_pathways,
    output_file=(
        all_immune_figures_path
        / "all_immune_cross_tissue_enrichment.png"
    ),
    title="Immune LR pathway enrichment across tissues"
)

Running pathway enrichment: Ctrl
Running pathway enrichment: EuE
Running pathway enrichment: EcO
Running pathway enrichment: EcP


# Analysis #2: Senescent/Dysfunctional Immune Communication

Cells labeled `SEN-high & DYS-high` are grouped as `SEN_DYS`. All remaining cells are grouped as `Other`.

The communication groups retain cell identity by combining the immune cell type and state label. Groups with fewer than 20 cells in a tissue are removed before running CellPhoneDB.

In future iterations, each group (Senescent, Dysfunctional, Senescent & Dysfunctional) can be analyzed and compared.

In [14]:
def add_sen_dys_groups(
    adata,
    state_col="sen_dysfunction_label",
    cell_type_col="cell_type_short"
):
    subset = adata.copy()

    subset.obs["sen_dys_group"] = np.where(
        subset.obs[state_col].astype(str)
        == "SEN-high & DYS-high",
        "SEN_DYS",
        "Other"
    )

    subset.obs["cell_type_state"] = (
        subset.obs[cell_type_col].astype(str)
        + "_"
        + subset.obs["sen_dys_group"].astype(str)
    ).astype("category")

    return subset


sen_dys_tissue_subsets = {
    tissue: add_sen_dys_groups(subset)
    for tissue, subset
    in all_cell_subsets.items()
}

sen_dys_subsets = run_cellphonedb_by_tissue(
    sen_dys_tissue_subsets,
    groupby="cell_type_state",
    use_raw=False,
    layer=(
        "counts"
        if "counts" in immune_subset.layers
        else None
    ),
    min_cells=MIN_CELLS
)

Running CellPhoneDB for Ctrl: 2,889 cells, 17 groups


/usr/local/lib/python3.12/dist-packages/legacy_api_wrap/__init__.py:88: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
/usr/local/lib/python3.12/dist-packages/liana/method/_pipe_utils/_pre.py:165: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
/usr/local/lib/python3.12/dist-packages/liana/method/_pipe_utils/_pre.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


Completed Ctrl: 51,878 tested interactions
Running CellPhoneDB for EuE: 9,294 cells, 24 groups


/usr/local/lib/python3.12/dist-packages/legacy_api_wrap/__init__.py:88: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
/usr/local/lib/python3.12/dist-packages/liana/method/_pipe_utils/_pre.py:165: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
/usr/local/lib/python3.12/dist-packages/liana/method/_pipe_utils/_pre.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


Completed EuE: 129,692 tested interactions
Running CellPhoneDB for EcO: 4,013 cells, 18 groups


/usr/local/lib/python3.12/dist-packages/legacy_api_wrap/__init__.py:88: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
/usr/local/lib/python3.12/dist-packages/liana/method/_pipe_utils/_pre.py:165: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
/usr/local/lib/python3.12/dist-packages/liana/method/_pipe_utils/_pre.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


Completed EcO: 56,009 tested interactions
Running CellPhoneDB for EcP: 11,483 cells, 21 groups


/usr/local/lib/python3.12/dist-packages/legacy_api_wrap/__init__.py:88: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
/usr/local/lib/python3.12/dist-packages/liana/method/_pipe_utils/_pre.py:165: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
/usr/local/lib/python3.12/dist-packages/liana/method/_pipe_utils/_pre.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


Completed EcP: 86,498 tested interactions


In [15]:
# -- SEN/DYS communication-group counts
for tissue, subset in sen_dys_subsets.items():

    print(f"\n{tissue}")

    display(
        subset.obs["cell_type_state"]
        .value_counts()
        .rename("cell_count")
        .to_frame()
    )


Ctrl


,cell_count
cell_type_state,
NK-CD16-_Other,622
TRM_Other,430
γδ T_Other,356
CD4 T_Other,323
Mono-C_Other,271
cDC2_Other,245
NK-CD16+_Other,225
B_Other,120
Mono-NC_Other,54



EuE


,cell_count
cell_type_state,
NK-CD16-_Other,2097
CD4 T_Other,1257
γδ T_Other,1199
Mono-C_Other,882
TRM_Other,740
B_Other,522
NK-CD16+_Other,476
cDC2_Other,351
γδ T_SEN_DYS,251



EcO


,cell_count
cell_type_state,
TRM_Other,1032
γδ T_Other,735
CD4 T_Other,603
NK-CD16-_Other,494
NK-CD16+_Other,227
Mono-C_Other,156
CD4 T_SEN_DYS,144
B_Other,126
cDC2_Other,111



EcP


,cell_count
cell_type_state,
Mono-C_Other,1876
γδ T_Other,1672
CD4 T_Other,1624
NK-CD16+_Other,1222
TRM_Other,1105
cDC2_Other,1027
NK-CD16-_Other,937
B_Other,458
CD4 T_SEN_DYS,234


In [16]:
# Tile plots
# -- Broad tile plots
for tissue, subset in sen_dys_subsets.items():

    tile_plot = li.pl.tileplot(
        adata=subset,
        fill="means",
        label="props",
        label_fun=lambda value: f"{value:.2f}",
        top_n=TOP_N,
        orderby="lr_means",
        orderby_ascending=False,
        filter_fun=lambda frame: (
            frame["cellphone_pvals"] < P_VALUE_THRESHOLD
        )
        & (
            frame["lr_means"] >= LR_MEANS_THRESHOLD
        ),
        figure_size=(20, 20)
    )

    tile_plot = (
        tile_plot
        + ggtitle(
            f"SEN/DYS versus Other CCC — {tissue}"
        )
    )

    save_liana_plot(
        tile_plot,
        output_path=sen_dys_figures_path,
        filename=f"{tissue}_sen_dys_tileplot.png",
        width=20,
        height=20
    )

/usr/local/lib/python3.12/dist-packages/liana/plotting/_common.py:108: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/usr/local/lib/python3.12/dist-packages/liana/plotting/_common.py:108: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/usr/local/lib/python3.12/dist-packages/liana/plotting/_common.py:108: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.or

Because all of these interactions are not that interesting and don't stand out, going to pull out some of the more interesting cell types to look at more in depth in the dot plot

In [18]:
# -- Focused dot plots
focused_panels = {
    "TRM": {
        "title": "TRM signaling",
        "sources": [
            "TRM_SEN_DYS",
            "TRM_Other",
        ],
        "targets": [
            "Mono-C_Other",
            "Mono-NC_Other",
            "cDC2_Other",
            "NK-CD16+_Other",
            "NK-CD16-_Other",
            "CD4 T_Other",
            "CD8 T_Other",
            "Treg_Other",
        ],
    },

    "cDC2": {
        "title": "cDC2 signaling",
        "sources": [
            "cDC2_SEN_DYS",
            "cDC2_Other",
        ],
        "targets": [
            "Mono-C_Other",
            "Mono-NC_Other",
            "TRM_Other",
            "NK-CD16+_Other",
            "NK-CD16-_Other",
            "CD4 T_Other",
            "CD8 T_Other",
            "Treg_Other",
        ],
    },

    "monocytes": {
        "title": "Monocyte signaling",
        "sources": [
            "Mono-C_SEN_DYS",
            "Mono-C_Other",
            "Mono-NC_SEN_DYS",
            "Mono-NC_Other",
        ],
        "targets": [
            "TRM_Other",
            "cDC2_Other",
            "NK-CD16+_Other",
            "NK-CD16-_Other",
            "CD4 T_Other",
            "CD8 T_Other",
            "Treg_Other",
        ],
    },

    "NK": {
        "title": "NK-cell signaling",
        "sources": [
            "NK-CD16+_SEN_DYS",
            "NK-CD16+_Other",
            "NK-CD16-_SEN_DYS",
            "NK-CD16-_Other",
        ],
        "targets": [
            "Mono-C_Other",
            "Mono-NC_Other",
            "TRM_Other",
            "cDC2_Other",
            "CD4 T_Other",
            "CD8 T_Other",
            "Treg_Other",
        ],
    },
}

for tissue, subset in sen_dys_subsets.items():

    liana_results = subset.uns["liana_res"]

    available_sources = set(
        liana_results["source"]
        .astype(str)
        .unique()
    )

    available_targets = set(
        liana_results["target"]
        .astype(str)
        .unique()
    )

    for panel_name, panel in focused_panels.items():

        source_types = [
            label
            for label in panel["sources"]
            if label in available_sources
        ]

        target_types = [
            label
            for label in panel["targets"]
            if label in available_targets
        ]

        print(f"\n{tissue} — {panel_name}")
        print("Sources:", source_types)
        print("Targets:", target_types)

        if not source_types or not target_types:
            print(
                f"Skipping {tissue} — {panel_name}: "
                "no valid focused source-target groups."
            )
            continue

        focused_dot_plot = li.pl.dotplot(
            adata=subset,
            colour="lr_means",
            size="cellphone_pvals",
            inverse_size=True,
            source_labels=source_types,
            target_labels=target_types,
            top_n=20,
            orderby="lr_means",
            orderby_ascending=False,
            filter_fun=lambda frame: (
                frame["cellphone_pvals"] < P_VALUE_THRESHOLD
            )
            & (
                frame["lr_means"] >= LR_MEANS_THRESHOLD
            ),
            figure_size=(15, 12)
        )

        focused_dot_plot = (
            focused_dot_plot
            + ggtitle(
                f"{panel['title']} — {tissue}"
            )
            + labs(
                colour="Mean expression",
                size="p-value"
            )
        )

        save_liana_plot(
            focused_dot_plot,
            output_path=sen_dys_figures_path,
            filename=(
                f"{tissue}_sen_dys_"
                f"{panel_name}_dotplot.png"
            ),
            width=15,
            height=12
        )


Ctrl — TRM
Sources: ['TRM_SEN_DYS', 'TRM_Other']
Targets: ['Mono-C_Other', 'Mono-NC_Other', 'cDC2_Other', 'NK-CD16+_Other', 'NK-CD16-_Other', 'CD4 T_Other', 'CD8 T_Other', 'Treg_Other']


/usr/local/lib/python3.12/dist-packages/liana/plotting/_common.py:108: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



Ctrl — cDC2
Sources: ['cDC2_SEN_DYS', 'cDC2_Other']
Targets: ['Mono-C_Other', 'Mono-NC_Other', 'TRM_Other', 'NK-CD16+_Other', 'NK-CD16-_Other', 'CD4 T_Other', 'CD8 T_Other', 'Treg_Other']


/usr/local/lib/python3.12/dist-packages/liana/plotting/_common.py:108: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



Ctrl — monocytes
Sources: ['Mono-C_SEN_DYS', 'Mono-C_Other', 'Mono-NC_Other']
Targets: ['TRM_Other', 'cDC2_Other', 'NK-CD16+_Other', 'NK-CD16-_Other', 'CD4 T_Other', 'CD8 T_Other', 'Treg_Other']


...

/usr/local/lib/python3.12/dist-packages/liana/plotting/_common.py:108: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



EcP — monocytes
Sources: ['Mono-C_SEN_DYS', 'Mono-C_Other', 'Mono-NC_Other']
Targets: ['TRM_Other', 'cDC2_Other', 'NK-CD16+_Other', 'NK-CD16-_Other', 'CD4 T_Other', 'CD8 T_Other', 'Treg_Other']


/usr/local/lib/python3.12/dist-packages/liana/plotting/_common.py:108: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



EcP — NK
Sources: ['NK-CD16+_SEN_DYS', 'NK-CD16+_Other', 'NK-CD16-_SEN_DYS', 'NK-CD16-_Other']
Targets: ['Mono-C_Other', 'Mono-NC_Other', 'TRM_Other', 'cDC2_Other', 'CD4 T_Other', 'CD8 T_Other', 'Treg_Other']


/usr/local/lib/python3.12/dist-packages/liana/plotting/_common.py:108: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


In [19]:
# -- Significant SEN/DYS interactions
significant_sen_dys = save_significant_results(
    sen_dys_subsets,
    output_path=sen_dys_results_path,
    file_suffix="sen_dys_significant_lr_pairs"
)

sen_dys_lr_genes = extract_lr_genes(
    significant_sen_dys
)

Ctrl: 21,000 significant interactions
EuE: 51,446 significant interactions
EcO: 18,492 significant interactions
EcP: 34,028 significant interactions
Ctrl: 549 unique LR genes
EuE: 658 unique LR genes
EcO: 589 unique LR genes
EcP: 611 unique LR genes


In [20]:
# -- SEN/DYS pathway enrichment
sen_dys_pathways = run_pathway_enrichment(
    sen_dys_lr_genes,
    output_path=sen_dys_pathway_path,
    file_suffix="sen_dys_pathway_enrichment"
)

plot_cross_tissue_enrichment(
    sen_dys_pathways,
    output_file=(
        sen_dys_figures_path
        / "sen_dys_cross_tissue_enrichment.png"
    ),
    title=(
        "Senescent/dysfunctional immune-cell "
        "LR pathway enrichment across tissues"
    )
)

Running pathway enrichment: Ctrl
Running pathway enrichment: EuE
Running pathway enrichment: EcO
Running pathway enrichment: EcP


In [21]:
def make_h5ad_safe(adata):
    """
    Convert Arrow-backed string columns to HDF5-safe pandas dtypes.
    """

    adata = adata.copy()

    for table_name in ["obs", "var"]:
        table = getattr(adata, table_name)

        for column in table.columns:
            array_type = type(table[column].array).__name__

            if "ArrowStringArray" in array_type:
                table[column] = (
                    table[column]
                    .astype("string")
                    .astype(object)
                )

        setattr(adata, table_name, table)

    return adata

In [22]:
# -- SAVEEE
all_immune_object_path = (
    project_dir
    / "data"
    / "interim"
    / "GSE179640"
    / "single_cell_ccc"
    / "all_immune"
)

sen_dys_object_path = (
    project_dir
    / "data"
    / "interim"
    / "GSE179640"
    / "single_cell_ccc"
    / "sen_dys"
)

all_immune_object_path.mkdir(
    parents=True,
    exist_ok=True
)

sen_dys_object_path.mkdir(
    parents=True,
    exist_ok=True
)

for tissue, subset in all_immune_subsets.items():

    subset_to_save = make_h5ad_safe(
        subset
    )

    subset_to_save.write_h5ad(
        all_immune_object_path
        / f"{tissue}_all_immune_ccc.h5ad"
    )

    print(
        f"Saved all-immune object: {tissue}"
    )


for tissue, subset in sen_dys_subsets.items():

    subset_to_save = make_h5ad_safe(
        subset
    )

    subset_to_save.write_h5ad(
        sen_dys_object_path
        / f"{tissue}_sen_dys_ccc.h5ad"
    )

    print(
        f"Saved SEN/DYS object: {tissue}"
    )

Saved all-immune object: Ctrl
Saved all-immune object: EuE
Saved all-immune object: EcO
Saved all-immune object: EcP
Saved SEN/DYS object: Ctrl
Saved SEN/DYS object: EuE
Saved SEN/DYS object: EcO
Saved SEN/DYS object: EcP
